In [6]:
import pandas as pd
import openai

In [7]:
df = pd.read_csv('lichess_preprocessed.csv')

df.head()

,event,site,white,white_elo,black,black_elo,result,termination,time_control,opening,moves
0,Rated Bullet game,https://lichess.org/rzhm0tmn,fil77,1401,PierreDaChello,1515,0-1,Normal,60+0,Horwitz Defense,1. d4 e6 2. f3 d6 3. h4 b6 4. g4 Bb7 5. e4 Nd7...
1,Rated Bullet game,https://lichess.org/mj65f4y8,fil77,1393,ivanbruno,1195,1-0,Time forfeit,60+0,Queen's Pawn Game,1. d4 d5 2. f3 e6 3. h4 a6 4. e4 h6 5. e5 Nc6 ...
2,Rated Bullet game,https://lichess.org/shvts5u8,fil77,1399,ivanbruno,1188,0-1,Normal,60+0,Queen's Pawn,1. d4 d6 2. f3 e5 3. e4 exd4 4. Qxd4 Nf6 5. c3...
3,Rated Bullet game,https://lichess.org/km2aynp4,fil77,1382,ivanbruno,1209,1-0,Normal,120+0,Queen's Pawn,1. d4 d6 2. f3 e5 3. e4 exd4 4. Qxd4 Nc6 5. Qd...
4,Rated Bullet game,https://lichess.org/kc8qnfhx,fil77,1388,ivanbruno,1202,1-0,Time forfeit,120+0,Queen's Pawn,1. d4 d6 2. f3 h6 3. h4 Nf6 4. g4 d5 5. g5 hxg...


In [8]:
import pandas as pd
import openai
import re

# Initialisation client OpenAI
client = openai.Client(
    api_key="sk-rc-L42_c4ej2zLysp2lc9ioKA",
    base_url="https://fmapi.swissai.cscs.ch"
)

# Fonction de détection de style
def detect_chess_style_from_moves(moves, side="white"):
    prompt = f"Given the following moves played by {side}, describe their chess style in less than 10 words:\n{moves}"
    try:
        res = client.chat.completions.create(
            model="meta-llama/Llama-3.3-70B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            stream=True,
        )
        response = ""
        for chunk in res:
            if len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                response += chunk.choices[0].delta.content
        return response.strip()
    except Exception as e:
        return f"Error: {e}"

# Fonction pour extraire les coups blancs et noirs d’un string PGN-like
def extract_white_black_moves(full_moves_str):
    tokens = re.findall(r"\d+\.\s*([^\s]+)\s+([^\s]+)", full_moves_str)
    white_moves = " ".join([w for w, b in tokens])
    black_moves = " ".join([b for w, b in tokens])
    return white_moves, black_moves


# Application des fonctions
def analyze_row(row):
    white, black = extract_white_black_moves(row["moves"])
    white_style = detect_chess_style_from_moves(white, "white")
    black_style = detect_chess_style_from_moves(black, "black")
    return pd.Series({"white_style": white_style, "black_style": black_style})

df[["white_style", "black_style"]] = df.apply(analyze_row, axis=1)

# Affichage
print(df[["white_style", "black_style"]])


                                           white_style  \
0    Aggressive and chaotic, prioritizing attack ov...   
1                       Aggressive and unconventional.   
2            Aggressive, yet chaotic and disorganized.   
3    Aggressive and reckless, prioritizing attack o...   
4                     Aggressive and somewhat chaotic.   
..                                                 ...   
994                  Aggressive and somewhat reckless.   
995                   Aggressive and somewhat chaotic.   
996                     Aggressive and unconventional.   
997              Aggressive and somewhat disorganized.   
998  Aggressive and tactical, with slow king develo...   

                                           black_style  
0                             Aggressive and tactical.  
1    Aggressive and tactical, with some reckless ab...  
2                             Aggressive and tactical.  
3    Aggressive and tactical, sacrificing material ...  
4    Aggressive an

In [9]:
df[["white_style", "black_style"]].head()

,white_style,black_style
0,"Aggressive and chaotic, prioritizing attack ov...",Aggressive and tactical.
1,Aggressive and unconventional.,"Aggressive and tactical, with some reckless ab..."
2,"Aggressive, yet chaotic and disorganized.",Aggressive and tactical.
3,"Aggressive and reckless, prioritizing attack o...","Aggressive and tactical, sacrificing material ..."
4,Aggressive and somewhat chaotic.,"Aggressive and tactical, favoring complex posi..."


In [10]:
df.to_csv('df_with_style.csv', index=False)